# 26. LangChain + OpenAI로 PDF 근거 답변 만들기

**오늘 순서:** 설정 확인 → PDF 근거 검색 → 전달할 내용 확인 → 답변 생성 → 원문 대조.

- 임베딩 모델은 기존 한국어 로컬 모델입니다. 질문과 비슷한 PDF 글을 찾습니다.
- 생성 모델 `gpt-6-luna`는 찾은 글을 읽기 쉬운 답변으로 정리합니다. 새 모델을 학습하는 과정은 아닙니다.
- `[모델 추가]` 사용자 선택으로 6 Luna를 적용했습니다. 확인한 수업 모델과 구별하며, 생성 모델만 바꾸므로 기존 임베딩·DB 자료는 그대로 사용합니다.
- `[프로젝트 추가]` 이번에는 `reasoning_effort='none'`과 `temperature=0`으로 기존 근거 정리 역할을 유지합니다. 추론 수준을 높일 때는 생성 옵션을 함께 검토해야 합니다.
- `[수업 개념]` 프롬프트, ChatOpenAI, StrOutputParser는 수업 LCEL 실습에서 확인한 역할입니다.
- `[프로젝트 추가]` 개인 DB 조회, 검토 상태, 복합 질문 분리, 출처 검사, 발췌 복귀 규칙은 싼타페용 처리입니다.
- `[프로젝트 추가]` 안전벨트 답변의 명시적 사용 금지 누락을 확인해, 질문과 관련된 사용 금지·필수 주의를 별도 답변 항목에 넣는 생성 지침을 추가했습니다. 코드 적용과 재실행 효과 확인은 구별합니다. 기존 근거·변경 전 답변을 보관하고 생성 모듈을 다시 불러온 뒤 같은 질문으로 비교합니다.
- 생성에 전달하는 자료는 확인한 PDF 글과 그림 설명입니다. 그림 픽셀은 전달하지 않습니다.
- DB 저장·문서 재임베딩·그림 업로드는 없습니다. 생성 셀을 실제 실행하면 OpenAI API 비용이 발생할 수 있습니다.

**우선 첫 코드 셀만 실행하세요.** 첫 셀은 설정만 확인하고 DB 검색·임베딩·API 호출을 하지 않습니다. 이번 노트북은 준비한 코드이며 실제 실행 결과는 아직 없습니다.

In [ ]:
from pathlib import Path
import sys

# [프로젝트 추가] notebooks 폴더에서 실행해도 프로젝트의 src 폴더를 찾습니다.
current_folder = Path.cwd().resolve()
project_folder = next((folder for folder in (current_folder, *current_folder.parents)
                       if (folder / 'pyproject.toml').is_file() and (folder / 'src').is_dir()), None)
if project_folder is None:
    raise FileNotFoundError('mle-02-p1-team1 프로젝트 안에서 노트북을 열어 주세요.')
src_path = str(project_folder / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from car_search_rag.zzong_santafe_lag.llm_answer_service import LlmManualAnswerService

# [프로젝트 적용] 이미 저장한 개인 전체 작업을 조회합니다. 팀원 테이블을 변경하지 않습니다.
run_id = 'ca9d2721-3d48-42a8-8748-3935e78515e5'
llm_service = LlmManualAnswerService(run_id)
settings = llm_service.settings()
print('답변 생성 모델:', settings['generation_model'])
print('추론 설정:', settings['reasoning_effort'])
print('API 키 설정 있음:', settings['api_key_configured'])
print('키가 유효한지는 아직 확인하지 않았습니다.')
print('답변 출력 한도:', settings['max_output_tokens'], '토큰')
print('준비 완료. 아직 DB 검색·임베딩·OpenAI 호출은 하지 않았습니다.')

## 1. 질문의 PDF 근거 찾기

다음 셀은 개인 DB를 읽고 질문만 로컬에서 임베딩합니다. **OpenAI는 호출하지 않습니다.**
처음에는 차대번호 질문 하나로 시작합니다. 이후 `question`을 복합 질문으로 바꿔 항목별 출처가 모두 있는지 살펴봅니다.

예: `엔진 오일 교체 시 주입량과 SAE 점도 표의 온도 눈금을 함께 알려줘.`

출처 번호는 PDF 실제 쪽수와 인쇄 매뉴얼 쪽수를 구분합니다. `partial_evidence`는 일부 항목의 근거가 없다는 뜻이고, `needs_review`는 원본 대조가 남았다는 뜻입니다.

In [ ]:
question = '차대번호가 새겨진 위치는 어디인가요?'

# [프로젝트 적용] 검색은 작은 자식 청크로 하고, 답변 근거는 부모 설명과 필수 각주를 가져옵니다.
# '함께/각각/둘 다'와 '과/와'가 있는 두세 항목은 규칙으로 나눠 검색합니다. 모든 문장을 분석하지는 못합니다.
evidence = llm_service.prepare_evidence(question, top_k=5, progress=print)
print('근거 상태:', evidence['status'])
print('나눠서 검색한 항목:', evidence.get('subquestions', [question]))
for source in evidence['sources']:
    print(source['label'], '|', source['title'])
print('확인 못한 항목:', evidence.get('missing_subquestions', []))
print('
PDF 발췌:
', evidence['answer'])

## 2. 모델에 보낼 내용 확인

프롬프트는 모델에게 주는 안내문입니다. 아래에서는 질문과 PDF 근거만 보냅니다. `ready_for_generation=True`는 검토 상태와 입력 길이가 준비 조건에 맞는다는 뜻이며 답변 정확도 판정은 아닙니다.

근거가 너무 길면 경고·각주를 잘라 보내지 않고 질문을 좁히도록 안내합니다. 이 셀도 API 호출은 없습니다.

In [ ]:
# [프로젝트 추가] API 호출 전에 전달할 PDF 근거를 직접 볼 수 있도록 준비·생성을 나눴습니다.
preview = llm_service.preview(evidence)
print('생성 준비 상태:', preview['ready_for_generation'])
print('입력 글자 수:', preview['prompt_characters'])
print('안내:', preview['reason'] or '준비한 근거로 답변 생성을 실행할 수 있습니다.')
print('
모델에 보낼 질문과 근거:
', preview['payload_text'])

## 3. OpenAI 답변 생성

내용을 확인한 후 **`run_generation = True`로 바꾸고 이 셀을 실행하면 API를 호출**합니다. 키를 채팅이나 노트북에 붙여 넣지 않습니다. 기존 설정을 읽습니다.

- 한 번의 생성 시도이며 자동 재시도는 없습니다. 셀을 다시 실행하면 새 호출입니다.
- 근거 미검토·부분 누락·입력 한도 초과이면 호출을 보류하고 기존 발췌를 유지합니다.
- 생성 실패·출력 잘림·출처 번호 오류이면 발췌로 돌아갑니다. 실패한 요청의 과금 여부를 이 코드가 확인하지는 않습니다.
- 정상 생성 상태는 `generated_answer`입니다. 출처 번호 검사와 실제 문장의 정확도 대조는 서로 다릅니다.

In [ ]:
run_generation = False
generated = None

# [수업 개념/프로젝트 적용] 내부에서 ChatPromptTemplate → ChatOpenAI → StrOutputParser 순서로 처리합니다.
# 프롬프트가 질문·근거를 만들고, 모델이 답변하며, parser가 모델 응답에서 글을 꺼냅니다.
if run_generation:
    generated = llm_service.generate(evidence)
    print('답변 상태:', generated['status'])
    print('OpenAI 호출 시도:', generated['llm_called'])
    print('생성 안내:', generated.get('generation_notice', ''))
    print('사용 토큰:', generated.get('token_usage'))
    print('
답변:
', generated['answer'])
else:
    print('아직 OpenAI를 호출하지 않았습니다. 위의 근거를 확인한 뒤 True로 바꿔 실행하세요.')

## 4. 답변과 근거 대조

생성 문장과 아래 원문을 비교해 숫자·단위·조건·경고·각주가 빠지거나 바뀌지 않았는지 확인합니다. 아래 `quote`는 검색용 전처리를 적용한 부모 글이며 `raw_text`가 보존한 원문입니다.

그림은 검색한 부모에 연결된 참조를 유지합니다. 공개 파일과 확인한 설명이 있는 그림만 표시 가능한 목록에 들어갑니다. 새 이미지 임베딩·시각 분석·업로드를 한 것은 아닙니다.

실제 실행 결과를 확인한 후 기존 10문항의 근거 개선과 생성 답변의 의미 정확도를 별도로 비교합니다. 이번 작성만으로 평가 완료나 M5 전체 완료로 표시하지 않습니다.

In [ ]:
# [프로젝트 추가] 생성으로 원문을 덮어쓰지 않아 언제든 수치·경고를 다시 대조할 수 있습니다.
if generated is None:
    print('생성 셀을 실행한 후 답변과 근거를 비교하세요.')
else:
    for source in generated['sources']:
        print('
', source['label'], source['title'])
        print('검색용 부모 글:
', source['quote'])
        print('보존 원문:
', source['raw_text'])
    print('
표시 가능한 그림 수:', len(generated['images']))
    for image in generated['images']:
        print('그림 출처: PDF', image['pdf_page_number'], '쪽 |', image['public_url'])
    print('업로드 또는 연결 확인이 남은 그림 수:', len(generated['pending_image_references']))